# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     |  Claudio Alejo Encarnación Martínez |
| **Fecha**      | 10 septiembre 2026  |
| **Expediente** |  750597 |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv), utiliza train-test-split de 70/30 y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/Advertising.csv')
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [ ]:
#Utiliza el dataset de publicidad (Advertising.csv), utiliza train-test-split de 70/30 y realiza 3 regresiones múltiples:
#$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$



In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score

X = df[['TV', 'radio', 'newspaper']]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

modelos = {
    'Sin penalización': LinearRegression()
}

for alpha in [0.01, 0.1, 1, 10, 100]:
    modelos[f'Ridge alpha={alpha}'] = Ridge(alpha=alpha)
    modelos[f'Lasso alpha={alpha}'] = Lasso(alpha=alpha, max_iter=10000)

resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    
    resultados.append({
        'Modelo': nombre,
        'Intercepto': modelo.intercept_,
        'TV': modelo.coef_[0],
        'radio': modelo.coef_[1],
        'newspaper': modelo.coef_[2],
        'R² entrenamiento': modelo.score(X_train, y_train),
        'R² prueba': modelo.score(X_test, y_test)
    })

resultados_df = pd.DataFrame(resultados)
resultados_df.round(4)

,Modelo,Intercepto,TV,radio,newspaper,R² entrenamiento,R² prueba
0,Sin penalización,2.7089,0.0441,0.1993,0.0069,0.9055,0.8609
1,Ridge alpha=0.01,2.7090,0.0441,0.1993,0.0069,0.9055,0.8609
2,Lasso alpha=0.01,2.7104,0.0441,0.1992,0.0069,0.9055,0.8610
3,Ridge alpha=0.1,2.7090,0.0441,0.1993,0.0069,0.9055,0.8609
4,Lasso alpha=0.1,2.7239,0.0441,0.1989,0.0067,0.9055,0.8614
5,Ridge alpha=1,2.7091,0.0441,0.1993,0.0069,0.9055,0.8610
6,Lasso alpha=1,2.8584,0.0440,0.1953,0.0055,0.9053,0.8651
7,Ridge alpha=10,2.7102,0.0441,0.1992,0.0069,0.9055,0.8610
8,Lasso alpha=10,4.0756,0.0432,0.1559,0.0000,0.8867,0.8791
9,Ridge alpha=100,2.7210,0.0441,0.1985,0.0071,0.9055,0.8613
